In [0]:
#read data from raw_data volume 
# Add audit columns --> 2 columns 
# define target
# write data to target table

#logging
'''
basic -- printing
'''

In [0]:
dbutils.library.restartPython()

In [0]:
dbutils.widgets.text("path", "")
config_path = dbutils.widgets.get("path")

In [0]:

# ============================================================
# 0. Libraries
# ============================================================

import pyspark.sql.functions as f
from common_utils.bronze import read_raw  , bronze_ingestor
from common_utils.logging import get_logger
import json
from datetime import datetime

# ============================================================
# 1. Setting up logger
# ============================================================

logger= get_logger("ds2b_sqlserver")

logger.info("Session started for sql_server")


# ============================================================
# 2. Config path
# ============================================================

dbutils.widgets.text("path", "")
config_path = dbutils.widgets.get("path")
logger.info("setting path for config files")

# ============================================================
# 3. Reading config path
# ============================================================
logger.info("Reading configuration from %s", config_path)
with open(config_path, "r") as file:
    config = json.load(file)

logger.info("configuration loaded succesfully")

# ============================================================
# 4. SOURCE / TARGET CONFIGURATION
# ============================================================

source_name = config["source"]["source_name"]
source_table = config["source"]["source_table"]
target_table = config["target"]["target_table"]
read_options = config["read_options"]
file_format = config["format"]["file_format"]



logger.info(
    "[%s] Source table: %s",
    source_name,
    source_table
)

logger.info(
    "[%s] Target table: %s",
    source_name,
    target_table
)

# ============================================================
# 5. Getting the date for latest file
# ============================================================

run_date = datetime.now().strftime("%Y-%m-%d")
logger.info("[%s] date data will be processed", run_date)

# ============================================================
# 6. buidling source path
# ============================================================
source_path = f"{config['source']['source_table']}/{config['source']['raw_folder']}/load_date={run_date}/"
# ============================================================
# 5. READ SOURCE TABLE
# ============================================================

logger.info(
    "[%s] Reading from %s",
    source_name,
    source_table
)

df = read_raw(spark, source_path, file_format, read_options)

logger.info(
    "[%s]  loaded successfully",
    source_name
)

# ============================================================
# 6. DISPLAY
# ============================================================

df.display()

# ============================================================
# 7. WRITE GOLD TABLE
# ============================================================
rows = bronze_ingestor(
        df,
        mode=config["write_options"]["mode"],
        target_table=target_table
    )



logger.info(
        "%s rows ingested into the bronze %s",
        rows,
        source_name
    )

